# Generative Models Workshop
## CMSC 178 - Introduction to Image Processing

In this workshop, you will explore generative models including autoencoders, VAEs, and GANs. You'll implement components of these models and see how they can generate new images.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Part 1: Understanding Autoencoders

Autoencoders learn to compress and reconstruct data. They consist of:
- **Encoder**: Compresses input to a latent representation
- **Latent space**: Compressed representation
- **Decoder**: Reconstructs input from latent representation

In [ ]:
# Load MNIST dataset
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

# Get a batch for visualization
dataiter = iter(train_loader)
images, labels = next(dataiter)

# Visualize some samples
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.axis('off')
    ax.set_title(f'Label: {labels[i].item()}')
plt.tight_layout()
plt.show()

### Activity 1: Complete the Autoencoder Architecture

Fill in the missing parts of the autoencoder below. Think about:
- What activation functions should be used?
- What should the output activation be?
- How do encoder and decoder mirror each other?

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super(Autoencoder, self).__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),  # TODO: What activation should we use here?
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),  # TODO: Fill in the decoder architecture
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 784),
            nn.Sigmoid()  # TODO: Why do we use Sigmoid here?
        )
    
    def forward(self, x):
        x = x.view(-1, 784)
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed.view(-1, 1, 28, 28)

# Initialize the model
autoencoder = Autoencoder(latent_dim=32).to(device)
print(autoencoder)

### Training the Autoencoder

In [ ]:
# Training setup
criterion = nn.MSELoss()
optimizer = optim.Adam(autoencoder.parameters(), lr=1e-3)

# Train for a few epochs
num_epochs = 5
losses = []

for epoch in range(num_epochs):
    epoch_loss = 0
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        
        # Forward pass
        reconstructed = autoencoder(data)
        loss = criterion(reconstructed, data)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(losses, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)
plt.show()

### Visualize Reconstructions

In [ ]:
# Get some test images
autoencoder.eval()
with torch.no_grad():
    test_images = images[:8].to(device)
    reconstructed = autoencoder(test_images).cpu()

# Visualize
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    # Original
    axes[0, i].imshow(test_images[i].cpu().squeeze(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Original', fontsize=12)
    
    # Reconstructed
    axes[1, i].imshow(reconstructed[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('Reconstructed', fontsize=12)

plt.tight_layout()
plt.show()

## Part 2: Variational Autoencoders (VAE)

VAEs learn a probabilistic latent space, allowing us to generate new samples by sampling from this space.

### Activity 2: Understand the VAE Loss Function

The VAE loss consists of two terms:
1. **Reconstruction loss**: How well can we reconstruct the input?
2. **KL divergence**: How close is our latent distribution to a standard normal?

Complete the loss function below:

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=20):
        super(VAE, self).__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU()
        )
        
        # Latent space
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 784),
            nn.Sigmoid()
        )
    
    def encode(self, x):
        h = self.encoder(x.view(-1, 784))
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        return self.decoder(z).view(-1, 1, 28, 28)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

def vae_loss(reconstructed, original, mu, logvar):
    # TODO: Implement reconstruction loss (MSE or BCE)
    recon_loss = nn.functional.mse_loss(reconstructed, original, reduction='sum')
    
    # TODO: Implement KL divergence
    # KL(N(mu, sigma^2) || N(0, 1)) = -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    return recon_loss + kl_loss

# Initialize VAE
vae = VAE(latent_dim=20).to(device)
vae_optimizer = optim.Adam(vae.parameters(), lr=1e-3)
print(vae)

### Train the VAE

In [ ]:
num_epochs = 5
vae_losses = []

for epoch in range(num_epochs):
    epoch_loss = 0
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        
        # Forward pass
        reconstructed, mu, logvar = vae(data)
        loss = vae_loss(reconstructed, data, mu, logvar)
        
        # Backward pass
        vae_optimizer.zero_grad()
        loss.backward()
        vae_optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader.dataset)
    vae_losses.append(avg_loss)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

plt.figure(figsize=(8, 4))
plt.plot(vae_losses, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('VAE Training Loss')
plt.grid(True)
plt.show()

### Activity 3: Generate New Samples

Now that we have a trained VAE, we can generate new images by sampling from the latent space!

In [ ]:
# TODO: Sample from the latent space and generate new images
vae.eval()
with torch.no_grad():
    # Sample from standard normal distribution
    z = torch.randn(16, 20).to(device)  # 16 samples, 20-dim latent space
    generated = vae.decode(z).cpu()

# Visualize generated samples
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i].squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle('Generated Samples from VAE', fontsize=14)
plt.tight_layout()
plt.show()

### Explore the Latent Space

Let's visualize how the latent space organizes different digits.

In [ ]:
# Encode test samples
vae.eval()
latents = []
labels_list = []

with torch.no_grad():
    for data, labels in train_loader:
        data = data.to(device)
        mu, _ = vae.encode(data)
        latents.append(mu.cpu().numpy())
        labels_list.append(labels.numpy())
        if len(latents) >= 10:  # Use subset for speed
            break

latents = np.concatenate(latents, axis=0)
labels_array = np.concatenate(labels_list, axis=0)

# Reduce to 2D using PCA
pca = PCA(n_components=2)
latents_2d = pca.fit_transform(latents)

# Plot
plt.figure(figsize=(10, 8))
scatter = plt.scatter(latents_2d[:, 0], latents_2d[:, 1], 
                     c=labels_array, cmap='tab10', alpha=0.5, s=10)
plt.colorbar(scatter, ticks=range(10))
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.title('VAE Latent Space (PCA Projection)')
plt.grid(True, alpha=0.3)
plt.show()

## Part 3: Introduction to GANs

Generative Adversarial Networks consist of two networks:
- **Generator**: Creates fake images from noise
- **Discriminator**: Distinguishes real from fake images

They are trained in an adversarial manner.

### Activity 4: Build a Simple GAN

Complete the generator and discriminator architectures below.

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100):
        super(Generator, self).__init__()
        
        # TODO: Build the generator
        # Input: latent_dim noise vector
        # Output: 28x28 image
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 784),
            nn.Tanh()  # Output in [-1, 1]
        )
    
    def forward(self, z):
        img = self.model(z)
        return img.view(-1, 1, 28, 28)

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        
        # TODO: Build the discriminator
        # Input: 28x28 image
        # Output: probability (real or fake)
        self.model = nn.Sequential(
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
    
    def forward(self, img):
        img_flat = img.view(-1, 784)
        validity = self.model(img_flat)
        return validity

# Initialize GAN
generator = Generator(latent_dim=100).to(device)
discriminator = Discriminator().to(device)

# Optimizers
g_optimizer = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Loss function
adversarial_loss = nn.BCELoss()

print("Generator:")
print(generator)
print("\nDiscriminator:")
print(discriminator)

### Activity 5: Implement GAN Training Loop

The GAN training alternates between:
1. Training the discriminator to distinguish real from fake
2. Training the generator to fool the discriminator

In [ ]:
# Prepare data (normalize to [-1, 1] for Tanh)
transform_gan = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])
train_dataset_gan = datasets.MNIST(root='./data', train=True, download=True, transform=transform_gan)
train_loader_gan = DataLoader(train_dataset_gan, batch_size=128, shuffle=True)

num_epochs = 10
latent_dim = 100

g_losses = []
d_losses = []

for epoch in range(num_epochs):
    for i, (real_images, _) in enumerate(train_loader_gan):
        batch_size = real_images.size(0)
        real_images = real_images.to(device)
        
        # Labels
        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)
        
        # ---------------------
        # Train Discriminator
        # ---------------------
        d_optimizer.zero_grad()
        
        # Loss on real images
        real_loss = adversarial_loss(discriminator(real_images), real_labels)
        
        # Generate fake images
        z = torch.randn(batch_size, latent_dim).to(device)
        fake_images = generator(z)
        
        # Loss on fake images
        fake_loss = adversarial_loss(discriminator(fake_images.detach()), fake_labels)
        
        # Total discriminator loss
        d_loss = real_loss + fake_loss
        d_loss.backward()
        d_optimizer.step()
        
        # -----------------
        # Train Generator
        # -----------------
        g_optimizer.zero_grad()
        
        # Generator wants discriminator to think fake images are real
        g_loss = adversarial_loss(discriminator(fake_images), real_labels)
        g_loss.backward()
        g_optimizer.step()
        
        if i % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i}/{len(train_loader_gan)}], '
                  f'D Loss: {d_loss.item():.4f}, G Loss: {g_loss.item():.4f}')
    
    g_losses.append(g_loss.item())
    d_losses.append(d_loss.item())

# Plot losses
plt.figure(figsize=(10, 4))
plt.plot(g_losses, label='Generator Loss')
plt.plot(d_losses, label='Discriminator Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('GAN Training Losses')
plt.legend()
plt.grid(True)
plt.show()

### Generate Images with GAN

In [ ]:
# Generate new images
generator.eval()
with torch.no_grad():
    z = torch.randn(16, latent_dim).to(device)
    generated_images = generator(z).cpu()
    # Denormalize from [-1, 1] to [0, 1]
    generated_images = (generated_images + 1) / 2

# Visualize
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated_images[i].squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle('GAN Generated Digits', fontsize=14)
plt.tight_layout()
plt.show()

## Reflection Questions

1. **Autoencoders vs VAEs**: What is the key difference between autoencoders and VAEs? Why does this make VAEs better for generation?

2. **VAE Latent Space**: Look at the PCA visualization of the VAE latent space. Are different digits well-separated? What does this tell you about the model?

3. **GAN Training**: Why is GAN training more difficult than training autoencoders or VAEs? What could go wrong?

4. **Quality Comparison**: Compare the quality of images generated by the VAE and GAN. Which produces better results? Why?

5. **Applications**: Think of three real-world applications for each type of generative model (autoencoders, VAEs, GANs).

## Extension Activities (Optional)

If you finish early, try these challenges:

1. **Interpolation**: Create smooth transitions between two digits by interpolating in the VAE latent space.

2. **Conditional Generation**: Modify the VAE or GAN to generate specific digits on demand (hint: add label information).

3. **Denoising**: Train an autoencoder to remove noise from images. Add Gaussian noise to MNIST and train the model to recover the original.

4. **Architecture Experiment**: Try different latent dimensions for the VAE. How does this affect generation quality and training?

5. **GAN Improvements**: Implement one of these GAN improvements:
   - Wasserstein GAN (WGAN)
   - Deep Convolutional GAN (DCGAN)
   - Label smoothing for the discriminator

In [ ]:
# Space for your extension activities
